# Network Intrusion Detection Using the NSL-KDD Dataset

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import urllib.request
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

def download_nsl_kdd_data():
    base_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
    files = {
        'train': 'KDDTrain+.txt',
        'test': 'KDDTest+.txt'
    }

    for dataset_type, filename in files.items():
        try:
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(base_url + filename, filename)
            print(f"Successfully downloaded {filename}")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
            return False
    return True

def load_nsl_kdd_data():
    column_names = [
        'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
        'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
        'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
        'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack_type', 'difficulty'
    ]

    try:
        train_data = pd.read_csv('KDDTrain+.txt', names=column_names)
        test_data = pd.read_csv('KDDTest+.txt', names=column_names)
        print("Data loaded successfully!")
        return train_data, test_data
    except:
        print("Files not found. Generating synthetic NSL-KDD-like data...")
        return generate_synthetic_nsl_kdd_data()

def generate_synthetic_nsl_kdd_data():
    np.random.seed(42)

    n_train = 25000
    n_test = 10000

    def create_synthetic_data(n_samples):
        data = []

        attack_types = ['normal', 'dos', 'probe', 'r2l', 'u2r']
        protocols = ['tcp', 'udp', 'icmp']
        services = ['http', 'smtp', 'ftp', 'ssh', 'telnet', 'domain_u', 'private']
        flags = ['SF', 'S0', 'REJ', 'RSTR', 'SH', 'RSTO']

        for i in range(n_samples):
            attack_type = np.random.choice(attack_types, p=[0.6, 0.2, 0.1, 0.05, 0.05])

            if attack_type == 'normal':
                duration = np.random.exponential(10)
                src_bytes = np.random.exponential(1000)
                dst_bytes = np.random.exponential(500)
                count = np.random.poisson(10)
                srv_count = np.random.poisson(8)
                serror_rate = np.random.beta(1, 10)
                rerror_rate = np.random.beta(1, 15)
                same_srv_rate = np.random.beta(8, 2)
                diff_srv_rate = np.random.beta(2, 8)
                dst_host_count = np.random.poisson(50)
                dst_host_srv_count = np.random.poisson(20)
            elif attack_type == 'dos':
                duration = np.random.exponential(1)
                src_bytes = np.random.exponential(50)
                dst_bytes = np.random.exponential(0.1)
                count = np.random.poisson(100)
                srv_count = np.random.poisson(80)
                serror_rate = np.random.beta(8, 2)
                rerror_rate = np.random.beta(5, 5)
                same_srv_rate = np.random.beta(9, 1)
                diff_srv_rate = np.random.beta(1, 9)
                dst_host_count = np.random.poisson(200)
                dst_host_srv_count = np.random.poisson(150)
            elif attack_type == 'probe':
                duration = np.random.exponential(0.5)
                src_bytes = np.random.exponential(10)
                dst_bytes = np.random.exponential(5)
                count = np.random.poisson(30)
                srv_count = np.random.poisson(5)
                serror_rate = np.random.beta(5, 5)
                rerror_rate = np.random.beta(3, 7)
                same_srv_rate = np.random.beta(3, 7)
                diff_srv_rate = np.random.beta(7, 3)
                dst_host_count = np.random.poisson(100)
                dst_host_srv_count = np.random.poisson(10)
            else:
                duration = np.random.exponential(5)
                src_bytes = np.random.exponential(200)
                dst_bytes = np.random.exponential(100)
                count = np.random.poisson(5)
                srv_count = np.random.poisson(3)
                serror_rate = np.random.beta(2, 8)
                rerror_rate = np.random.beta(2, 8)
                same_srv_rate = np.random.beta(5, 5)
                diff_srv_rate = np.random.beta(5, 5)
                dst_host_count = np.random.poisson(20)
                dst_host_srv_count = np.random.poisson(10)

            row = {
                'duration': max(0, duration),
                'protocol_type': np.random.choice(protocols),
                'service': np.random.choice(services),
                'flag': np.random.choice(flags),
                'src_bytes': max(0, src_bytes),
                'dst_bytes': max(0, dst_bytes),
                'land': np.random.choice([0, 1], p=[0.99, 0.01]),
                'wrong_fragment': np.random.poisson(0.1),
                'urgent': np.random.poisson(0.05),
                'hot': np.random.poisson(0.5),
                'num_failed_logins': np.random.poisson(0.1),
                'logged_in': np.random.choice([0, 1], p=[0.3, 0.7]),
                'num_compromised': np.random.poisson(0.1),
                'root_shell': np.random.choice([0, 1], p=[0.95, 0.05]),
                'su_attempted': np.random.choice([0, 1], p=[0.98, 0.02]),
                'num_root': np.random.poisson(0.2),
                'num_file_creations': np.random.poisson(0.3),
                'num_shells': np.random.poisson(0.1),
                'num_access_files': np.random.poisson(0.2),
                'num_outbound_cmds': 0,
                'is_host_login': np.random.choice([0, 1], p=[0.99, 0.01]),
                'is_guest_login': np.random.choice([0, 1], p=[0.95, 0.05]),
                'count': max(1, int(count)),
                'srv_count': max(1, int(srv_count)),
                'serror_rate': np.clip(serror_rate, 0, 1),
                'srv_serror_rate': np.clip(serror_rate + np.random.normal(0, 0.1), 0, 1),
                'rerror_rate': np.clip(rerror_rate, 0, 1),
                'srv_rerror_rate': np.clip(rerror_rate + np.random.normal(0, 0.1), 0, 1),
                'same_srv_rate': np.clip(same_srv_rate, 0, 1),
                'diff_srv_rate': np.clip(diff_srv_rate, 0, 1),
                'srv_diff_host_rate': np.clip(np.random.beta(2, 8), 0, 1),
                'dst_host_count': max(1, int(dst_host_count)),
                'dst_host_srv_count': max(1, int(dst_host_srv_count)),
                'dst_host_same_srv_rate': np.clip(np.random.beta(6, 4), 0, 1),
                'dst_host_diff_srv_rate': np.clip(np.random.beta(3, 7), 0, 1),
                'dst_host_same_src_port_rate': np.clip(np.random.beta(4, 6), 0, 1),
                'dst_host_srv_diff_host_rate': np.clip(np.random.beta(2, 8), 0, 1),
                'dst_host_serror_rate': np.clip(serror_rate + np.random.normal(0, 0.05), 0, 1),
                'dst_host_srv_serror_rate': np.clip(serror_rate + np.random.normal(0, 0.08), 0, 1),
                'dst_host_rerror_rate': np.clip(rerror_rate + np.random.normal(0, 0.05), 0, 1),
                'dst_host_srv_rerror_rate': np.clip(rerror_rate + np.random.normal(0, 0.08), 0, 1),
                'attack_type': attack_type,
                'difficulty': np.random.randint(1, 22)
            }
            data.append(row)

        return pd.DataFrame(data)

    train_data = create_synthetic_data(n_train)
    test_data = create_synthetic_data(n_test)

    print(f"Generated synthetic data - Train: {len(train_data)}, Test: {len(test_data)}")
    return train_data, test_data

def preprocess_data(train_data, test_data):
    train_data = train_data.copy()
    test_data = test_data.copy()

    train_data = train_data.drop('difficulty', axis=1)
    test_data = test_data.drop('difficulty', axis=1)

    binary_mapping = {'normal': 0}
    for col in ['dos', 'probe', 'r2l', 'u2r']:
        binary_mapping[col] = 1

    train_data['binary_label'] = train_data['attack_type'].map(binary_mapping)
    test_data['binary_label'] = test_data['attack_type'].map(binary_mapping)

    train_data['binary_label'] = train_data['binary_label'].fillna(1)
    test_data['binary_label'] = test_data['binary_label'].fillna(1)

    categorical_columns = ['protocol_type', 'service', 'flag']

    combined_data = pd.concat([train_data, test_data], ignore_index=True)

    label_encoders = {}
    for col in categorical_columns:
        le = LabelEncoder()
        combined_data[col + '_encoded'] = le.fit_transform(combined_data[col].astype(str))
        label_encoders[col] = le

    train_size = len(train_data)
    train_encoded = combined_data[:train_size].copy()
    test_encoded = combined_data[train_size:].copy()

    feature_columns = [col for col in train_encoded.columns if col not in
                      ['attack_type', 'binary_label'] + categorical_columns]

    scaler = StandardScaler()
    train_encoded[feature_columns] = scaler.fit_transform(train_encoded[feature_columns])
    test_encoded[feature_columns] = scaler.transform(test_encoded[feature_columns])

    return train_encoded, test_encoded, feature_columns, scaler, label_encoders

def create_neural_network(input_dim, num_classes=2):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(16, activation='relu'),
        Dropout(0.2),

        Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
    ])

    return model

print("Setting up NSL-KDD Intrusion Detection System...")

if not download_nsl_kdd_data():
    print("Using synthetic data generation...")

train_data, test_data = load_nsl_kdd_data()

print(f"\nDataset Overview:")
print(f"Training samples: {len(train_data)}")
print(f"Testing samples: {len(test_data)}")
print(f"Features: {train_data.shape[1]}")

print(f"\nAttack type distribution in training data:")
print(train_data['attack_type'].value_counts())

train_processed, test_processed, feature_columns, scaler, label_encoders = preprocess_data(train_data, test_data)

X_train = train_processed[feature_columns].values
y_train = train_processed['binary_label'].values
X_test = test_processed[feature_columns].values
y_test = test_processed['binary_label'].values

print(f"\nProcessed data shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

print(f"\nClass distribution:")
print(f"Training - Normal: {np.sum(y_train == 0)}, Attack: {np.sum(y_train == 1)}")
print(f"Testing - Normal: {np.sum(y_test == 0)}, Attack: {np.sum(y_test == 1)}")

models = {}

print("\n" + "="*50)
print("1. Training Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
models['Random Forest'] = (rf, rf_pred)

print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred):.4f}")

print("\n" + "="*50)
print("2. Training Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
models['Logistic Regression'] = (lr, lr_pred)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_pred):.4f}")

print("\n" + "="*50)
print("3. Training Neural Network...")

nn_model = create_neural_network(X_train.shape[1], 2)
nn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
]

history = nn_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

nn_pred_proba = nn_model.predict(X_test)
nn_pred = np.argmax(nn_pred_proba, axis=1)
models['Neural Network'] = (nn_model, nn_pred)

print(f"Neural Network Accuracy: {accuracy_score(y_test, nn_pred):.4f}")

plt.figure(figsize=(20, 15))

plt.subplot(3, 4, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Neural Network Training History - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(3, 4, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Neural Network Training History - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

feature_importance = rf.feature_importances_
top_features_idx = np.argsort(feature_importance)[-15:]
plt.subplot(3, 4, 3)
plt.barh(range(15), feature_importance[top_features_idx])
plt.yticks(range(15), [feature_columns[i] for i in top_features_idx])
plt.title('Top 15 Feature Importance (Random Forest)')
plt.xlabel('Importance')

plt.subplot(3, 4, 4)
attack_dist = train_data['attack_type'].value_counts()
plt.pie(attack_dist.values, labels=attack_dist.index, autopct='%1.1f%%')
plt.title('Attack Type Distribution')

for idx, (model_name, (model, predictions)) in enumerate(models.items()):
    cm = confusion_matrix(y_test, predictions)

    plt.subplot(3, 4, 5 + idx)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.subplot(3, 4, 8)
accuracies = [accuracy_score(y_test, pred) for _, (_, pred) in models.items()]
model_names = list(models.keys())
bars = plt.bar(model_names, accuracies, color=['skyblue', 'lightgreen', 'lightcoral'])
plt.title('Model Comparison - Accuracy')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

correlation_matrix = train_processed[feature_columns[:20]].corr()
plt.subplot(3, 4, 9)
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix (Top 20)')

plt.subplot(3, 4, 10)
normal_samples = train_processed[train_processed['binary_label'] == 0]['duration']
attack_samples = train_processed[train_processed['binary_label'] == 1]['duration']
plt.hist(normal_samples, bins=50, alpha=0.7, label='Normal', density=True)
plt.hist(attack_samples, bins=50, alpha=0.7, label='Attack', density=True)
plt.xlabel('Duration')
plt.ylabel('Density')
plt.title('Duration Distribution by Class')
plt.legend()
plt.yscale('log')

plt.subplot(3, 4, 11)
normal_bytes = train_processed[train_processed['binary_label'] == 0]['src_bytes']
attack_bytes = train_processed[train_processed['binary_label'] == 1]['src_bytes']
plt.scatter(normal_bytes[:1000], normal_samples[:1000], alpha=0.5, label='Normal', s=10)
plt.scatter(attack_bytes[:1000], attack_samples[:1000], alpha=0.5, label='Attack', s=10)
plt.xlabel('Source Bytes')
plt.ylabel('Duration')
plt.title('Duration vs Source Bytes')
plt.legend()
plt.xscale('log')
plt.yscale('log')

proba_data = []
for model_name, (model, _) in models.items():
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = nn_pred_proba[:, 1]
    proba_data.append(proba)

plt.subplot(3, 4, 12)
for i, (model_name, proba) in enumerate(zip(models.keys(), proba_data)):
    plt.hist(proba, bins=50, alpha=0.7, label=model_name, density=True)
plt.xlabel('Attack Probability')
plt.ylabel('Density')
plt.title('Attack Probability Distribution')
plt.legend()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORTS")
print("="*70)

for model_name, (model, predictions) in models.items():
    print(f"\n{model_name}:")
    print("-" * 50)
    print(classification_report(y_test, predictions,
                              target_names=['Normal', 'Attack']))

def detect_intrusion(features, model_name='Random Forest'):
    model, _ = models[model_name]
    features_scaled = scaler.transform([features])

    if hasattr(model, 'predict_proba'):
        probability = model.predict_proba(features_scaled)[0]
        prediction = model.predict(features_scaled)[0]
        return prediction, probability
    else:
        probability = nn_model.predict(features_scaled)[0]
        prediction = np.argmax(probability)
        return prediction, probability

print(f"\n" + "="*50)
print("Testing on sample data...")

sample_indices = np.random.choice(len(X_test), 5, replace=False)
for i, idx in enumerate(sample_indices):
    actual = y_test[idx]
    features = X_test[idx]

    print(f"\nSample {i+1}:")
    print(f"Actual: {'Attack' if actual == 1 else 'Normal'}")

    for model_name in models.keys():
        pred, prob = detect_intrusion(features, model_name)
        pred_label = 'Attack' if pred == 1 else 'Normal'
        confidence = prob[1] if len(prob) > 1 else prob
        print(f"{model_name}: {pred_label} (confidence: {confidence:.3f})")

print(f"\nModel components saved for future use!")
print("="*70)